# 04 MLM předtrénink (úkoly 2.9–2.11, E2)

Natrénuje jeden model podle konfigurace z `configs/pretrain/`. Stav se průběžně ukládá
na Drive (`checkpoints/<název>/`). Když Colab spadne nebo vyprší čas, stačí notebook
spustit znovu: trénink pokračuje od posledního uloženého kroku.

Runtime: **GPU** (A100 nebo L4). Testy úniku (`leak_*`) zvládne i T4.

| Konfigurace | Co dělá |
| --- | --- |
| `S_nucleotide`, `S_kmer6`, `S_bpe_600`, `S_bpe_4096` | E2: model S s danou tokenizací |
| `leak_kmer3_overlap`, `leak_kmer3` | test úniku na náhodné DNA (překrývající vs. nepřekrývající 3-mery) |
| `leak_nucleotide`, `leak_kmer6`, `leak_bpe_4096`, `leak_kmer6_overlap` | test na náhodné DNA pro hlavní tokenizace |

## 1. Nastavení

In [ ]:
WINDOW = 1024
CONFIG = "leak_kmer3_overlap"   # název souboru v configs/pretrain/ bez .yaml

# Přepsání hodnot z konfigurace, např. pro pilotní běh:
# OVERRIDES = ["train.budget_nt=2e8", "train.batch_size=64"]
OVERRIDES = []

## Prostředí

Funguje v Colabu (prohlížeč i VS Code) i lokálně. V Colabu připojí Google Drive,
stáhne kód z GitHubu a zkopíruje genom na rychlý lokální disk. Jinde se cesty
řídí proměnnou `DNALM_ROOT`. Všechny cesty jsou v objektu `P` (`P.fasta`,
`P.windows("train")`, `P.tokenizer("bpe_600")`, `P.figures`, ...).

Ve VS Code s Colab rozšířením: pokud se Drive nepřipojí, použij příkaz
`Colab: Mount Google Drive to Server...` a buňku spusť znovu.

In [ ]:
# Úvodní buňka (stejná ve všech notebookách, neměnit).
# Colab: stáhne aktuální kód z GitHubu. Lokálně (Mac, školní GPU): použije nainstalované repo.
import importlib.util, sys
if importlib.util.find_spec("google.colab"):
    !git clone -q https://github.com/Valasik0/dna-lm.git /content/dna-lm 2>/dev/null || git -C /content/dna-lm pull -q
    !git -C /content/dna-lm log --oneline -1
    if "/content/dna-lm/src" not in sys.path:
        sys.path.insert(0, "/content/dna-lm/src")

from dnalm.utils.colab import setup
P = setup(window=WINDOW, genome=True)

## 2. GPU a konfigurace

In [ ]:
import torch, yaml
from dnalm.pretrain import load_run_config

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "není (CPU/MPS)")
cfg = load_run_config(str(P.configs / "pretrain" / f"{CONFIG}.yaml"), OVERRIDES)
print(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

## 3. Trénink

Přerušený běh (pád Colabu, vypršení času) pokračuje od posledního checkpointu.

In [ ]:
from dnalm.pretrain import train

result = train(cfg, P=P)
result

## 4. Průběh tréninku

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dnalm.utils.plotting import apply_style, save_figure
apply_style()

m = pd.read_csv(P.checkpoints / cfg["name"] / "metrics.csv")
tr, va = m[m.split == "train"], m[m.split == "val"]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(tr.seen_nt / 1e9, tr.bpn, color="#9b9a95", linewidth=1.2, label="trénink (80/10/10)")
ax.plot(va.seen_nt / 1e9, va.bpn, color="#2a78d6", marker="o", markersize=4, label="validace")
if cfg["data"].get("source") == "random":
    ax.axhline(2.0, color="#52514e", linestyle="--", linewidth=1)
    ax.annotate("náhoda: 2 bity", (0, 2.0), xytext=(4, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("viděné nukleotidy (miliardy)")
ax.set_ylabel("bity na nukleotid")
ax.set_title(f"{cfg['name']}: průběh předtréninku", loc="left")
ax.legend()
save_figure(fig, f"pretrain_{cfg['name']}", str(P.figures))
plt.show()
va.tail()